# Ordinary Differential Equations

In [1]:
using SymPy

## 1. Separation of Variables

Consider the following example of an initial value problem:

$$
    \frac{dy}{dx} = \frac{x}{y}, \quad y(3) = 5.
$$

We first formulate this problem with SymPy.

The indepedent variable $x$ has to be declared as a symbol.

In [2]:
x = Sym("x")

x

The dependent variable $y$ is a symbolic function.

In [3]:
y = SymFunction("y")

y

To define the equation, we must use the ``Eq()``.

In [4]:
ode = Eq(diff(y(x), x), x/y(x))

d           x  
--(y(x)) = ----
dx         y(x)

We solve the ``ode`` with ``dsolve``.

In [5]:
sol = dsolve(ode)

2-element Vector{Sym{PyCall.PyObject}}:
 Eq(y(x), -sqrt(C1 + x^2))
  Eq(y(x), sqrt(C1 + x^2))

sys:1: SymPyDeprecationWarning: 

non-Expr objects in a Matrix is deprecated. Matrix represents
a mathematical matrix. To represent a container of non-numeric
entities, Use a list of lists, TableForm, NumPy array, or some
other data structure instead.

See https://docs.sympy.org/latest/explanation/active-deprecations.html#deprecated-non-expr-in-matrix
for details.

This has been deprecated since SymPy version 1.9. It
will be removed in a future version of SymPy.



We see there are two solutions, depending on a parameter $C_1$.

The initial condition$y (3) = 5$ has not yet been used.  For that, we need a dictionary to define the initial condition.

In [6]:
initcond = Dict(y(3) => 5)

Dict{Sym{PyCall.PyObject}, Int64} with 1 entry:
  y(3) => 5

In [7]:
solivp = dsolve(ode, ics=initcond)

          _________
         /  2      
y(x) = \/  x  + 16 

Now we see that we do have a unique solution.

## 2. ODEs to for Three Problems

We apply the scientific method to three problems.

### slowing down

*An attack submarine is cruising at 40 knots at a depth of 1000 feet
when suddenly the reactor scrams. After 1 minute way has dropped to 30 knots.  How long does the crew have to make repairs before forward motion falls below steerageway of 2 knots?*

Newton's rule equates the sum of the inertial forces with the sum of the external forces.

The inertial force is

$$
   F = m a,
$$

where $m$ is the mass and $a$ the acceleration.

The external force is the water resistance

$$
    R = -k v^2
$$

proportional to the square of the velocity, for some constant $k$.

Note that acceleration is the derivative of the velocity.

In [8]:
m, k, t = Sym("m, k, t")
v = SymFunction("v")
ode = Eq(m*diff(v(t),t), -k*v(t)^2)

  d              2   
m*--(v(t)) = -k*v (t)
  dt                 

Let lump the constants $m$ and $k$ into one constant, dividing by $m$ and replacing $k/m$ by $K$.

In [9]:
lefteq = ode.lhs()/m

d       
--(v(t))
dt      

In [10]:
K = Sym("K")
righteq = subs(ode.rhs()/m, k/m=>K)

    2   
-K*v (t)

In [11]:
newode = Eq(lefteq, righteq)

d              2   
--(v(t)) = -K*v (t)
dt                 

Recall that the *submarine is cruising at 40 knots* so
the initial condition is $v(0) = 40$.

In [12]:
initcond = Dict(v(0)=>40)

Dict{Sym{PyCall.PyObject}, Int64} with 1 entry:
  v(0) => 40

In [13]:
sol = dsolve(newode, ics=initcond)

           1     
v(t) = ----------
       K*t + 1/40

Recall that *after one minute way has dropped to 30 knots*, so
we determine $K$ using $v(1) = 30$.

In [14]:
Kequ = subs(sol, t=>1, v(1) => 30)

        1    
30 = --------
     K + 1/40

In [15]:
Ksol = solve(Kequ, K)

1-element Vector{Sym{PyCall.PyObject}}:
 1/120

In [16]:
vsol = subs(sol, K=>Ksol[1])

          1    
v(t) = --------
        t    1 
       --- + --
       120   40

The original equation was *how long before the steerageway fall below 2 knots?*

In [17]:
vequ = subs(vsol, v(t) => 2)

       1    
2 = --------
     t    1 
    --- + --
    120   40

In [18]:
solve(vequ, t)

1-element Vector{Sym{PyCall.PyObject}}:
 57

At 57 minutes the speed has dropped to 2 knots.  So the crew has 56 minutes left.

### cooling off

*A house furnace fails on a cold winter's evening when the outside (ambient) temperature of 20 degrees Fahrenheit. Although initially at 70 degree Fahrenheit, the inside temperature has fallen to 65 degrees Fahrenheit after 1 hour.
How long before the inside temperature reaches the damaging temperature of 32 degrees Fahrenheit?*

Newton's law of cooling states that the rate at which the temperature changes is proportional to the temperature gradient driving out the heat.

Let $T(t)$ be the temperature at time $t$.

In [19]:
T = SymFunction("T")
ode = Eq(diff(T(t),t), k*(T(t) - 20))

d                       
--(T(t)) = k*(T(t) - 20)
dt                      

The right hand side of the equation $T(t) - 20$ represents the difference of the current temperature $T(t)$ with the outside temperature of 20 degrees.

At $t=0$, the temperature is 70 degrees.  So we formulate the initial condition.

In [20]:
initcond = Dict(T(0) => 70)

Dict{Sym{PyCall.PyObject}, Int64} with 1 entry:
  T(0) => 70

In [21]:
sol = dsolve(ode, ics=initcond)

           k*t     
T(t) = 50*e    + 20

What is the constant $k$?  Recall that *the inside temperature has fallen to 65 degrees Fahrenheit, after 1 hour.* 

In [22]:
kequ = subs(sol, t=>1, T(1)=>65)

         k     
65 = 50*e  + 20

In [23]:
kval = solve(kequ, k)

1-element Vector{Sym{PyCall.PyObject}}:
 log(9/10)

In [24]:
Tsol = subs(sol, k=>kval[1])

           t*log(9/10)     
T(t) = 50*e            + 20

What is the original question again?  *How long before the inside temperature reaches the damaging temperature of 32 degrees Fahrenheit?*

In [25]:
Tequ = subs(Tsol, T(t)=>32)

         t*log(9/10)     
32 = 50*e            + 20

In [26]:
tval = solve(Tequ, t)

1-element Vector{Sym{PyCall.PyObject}}:
 log((6/25)^(1/log(9/10)))

To interpret the result, we convert to a 64-bit floating-point approximation.

In [27]:
Float64(tval[1])

13.545077553292497

So it takes about 13 and a half hour before it starts freezing inside.

### population modeling

*Predict the future population of a developed country.*

Considering the births, the growth of the population if proportional to its size.

Let $P(t)$ be the size of population at time $t$.

In [28]:
P = SymFunction("P")
ode = Eq(diff(P(t),t), k*P(t))

d                
--(P(t)) = k*P(t)
dt               

In [29]:
dsolve(ode)

           k*t
P(t) = C1*e   

The exponential growth is not realistic.  Deaths occur.

For a better model, consider the correction term, similar to the second term in a Taylor series.  This second term is proportional to the square of the population size.

In [30]:
logistics = Eq(diff(P(t),t), k*P(t) - K*(P(t))^2)

d               2            
--(P(t)) = - K*P (t) + k*P(t)
dt                           

This model is called the *logistics equation*.

In [31]:
Psol = dsolve(logistics)

             k*(C1 + t)   
          k*e             
P(t) = -------------------
         / k*(C1 + t)    \
       K*\e           - 1/

To interpret this model, let us normalize the size of the population at the beginning of time to one.

In [32]:
initcond = Dict(P(0)=>1)
Psol1 = dsolve(logistics, ics=initcond)

               /       /  K  \\   
               |    log|-----||   
               |       \K - k/|   
             k*|t + ----------|   
               \        k     /   
          k*e                     
P(t) = ---------------------------
         /   /       /  K  \\    \
         |   |    log|-----||    |
         |   |       \K - k/|    |
         | k*|t + ----------|    |
         |   \        k     /    |
       K*\e                   - 1/

Fortunately, there is the ``simplify`` we can apply to an expression. 

In [33]:
Psol2 = simplify(Psol1.rhs())

       k*t    
    k*e       
--------------
   k*t        
K*e    - K + k

Both constants $k$ and $K$ are positive.  Let us divide numerator and denominator by ``exp(k*t)``.

In [34]:
Psol3 = k/(K + (k-K)*exp(-k*t))

        k         
------------------
              -k*t
K + (-K + k)*e    

In the limit, as $t$ goes to $\infty$, the size of the population reaches $k/K$.  The $k/K$ is called the *carrying capacity* of the society.